In [128]:
from google.colab import files
uploaded = files.upload()

Saving employees.csv to employees.csv
Saving logins.txt to logins.txt
Saving orders.json to orders.json
Saving sales.csv to sales.csv


In [129]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("PySpark Exercise").getOrCreate()
print("Spark Session Created")

Spark Session Created


File Uploading

Dataset 1 — TXT (Website Login Logs)

Questions (TXT Processing)

1. Read the TXT file and print all names.

In [179]:
logins_df = spark.read.text("logins.txt")
logins_df = logins_df.withColumnRenamed("value", "name")
print("All names:")
logins_df.show()


All names:
+-----+
| name|
+-----+
|Rahul|
|Sneha|
|Arjun|
|Rahul|
|Priya|
|Sneha|
|Rahul|
|Karan|
|Arjun|
|Sneha|
|Rahul|
| Amit|
|Priya|
|Karan|
|Sneha|
|Rahul|
|Meera|
|Arjun|
|Sneha|
|Rahul|
+-----+
only showing top 20 rows


2. Count the total login events.

In [132]:
print("Total login events:")
print(logins_df.count())

Total login events:
26


3. Find the unique users.

In [137]:
print("Unique users:")
logins_df.select("name").distinct().show()

Unique users:
+-----+
| name|
+-----+
|Meera|
|Sneha|
|Priya|
|Rahul|
|Arjun|
| Amit|
|Karan|
+-----+



4. Count how many times each user logged in.

In [138]:
print("Login count per user:")
login_counts = logins_df.groupBy("name").count()
login_counts.show()

Login count per user:
+-----+-----+
| name|count|
+-----+-----+
|Meera|    1|
|Sneha|    6|
|Priya|    3|
|Rahul|    7|
|Arjun|    4|
| Amit|    2|
|Karan|    3|
+-----+-----+



5. Find the top 3 most active users.

In [139]:
print("Top 3 most active users:")
login_counts.orderBy("count", ascending=False).show(3)

Top 3 most active users:
+-----+-----+
| name|count|
+-----+-----+
|Rahul|    7|
|Sneha|    6|
|Arjun|    4|
+-----+-----+
only showing top 3 rows


6. Find users who logged in more than 4 times.

In [140]:
from pyspark.sql.functions import col

print("Users who logged in more than 4 times:")
login_counts.filter(col("count") > 4).show()

Users who logged in more than 4 times:
+-----+-----+
| name|count|
+-----+-----+
|Sneha|    6|
|Rahul|    7|
+-----+-----+



7. Convert the login list into a dictionary of counts.

In [141]:
login_dict = {}

for row in login_counts.collect():
    login_dict[row["name"]] = row["count"]

print(login_dict)

{'Meera': 1, 'Sneha': 6, 'Priya': 3, 'Rahul': 7, 'Arjun': 4, 'Amit': 2, 'Karan': 3}


Expected format example:

{
"Rahul": 7,
"Sneha": 6,
"Arjun": 4
}

Dataset 2 — CSV (Employees Dataset)

Questions (CSV Processing)

1. Load the CSV file.

In [142]:
employees_df = spark.read.csv("employees.csv", header=True, inferSchema=True)
print(" Employees data:")
employees_df.show()

 Employees data:
+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     1| Rahul|        IT| 70000|Hyderabad|
|     2| Sneha|        HR| 60000|Bangalore|
|     3| Arjun|        IT| 75000|  Chennai|
|     4| Priya|   Finance| 80000|Hyderabad|
|     5| Karan|        IT| 50000|   Mumbai|
|     6|  Amit|        HR| 58000|    Delhi|
|     7| Meera|   Finance| 82000|Bangalore|
|     8|  Ravi|        IT| 72000|Hyderabad|
|     9|  Neha|        HR| 61000|  Chennai|
|    10|Vikram|   Finance| 90000|    Delhi|
+------+------+----------+------+---------+



2. Count total employees.

In [143]:
print(" Total employees:")
print(employees_df.count())

 Total employees:
10


3. Show employees from IT department.

In [144]:
print("Employees from IT department:")
employees_df.filter((employees_df.department) == "IT").show()

Employees from IT department:
+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     1|Rahul|        IT| 70000|Hyderabad|
|     3|Arjun|        IT| 75000|  Chennai|
|     5|Karan|        IT| 50000|   Mumbai|
|     8| Ravi|        IT| 72000|Hyderabad|
+------+-----+----------+------+---------+



4. Find employees with salary greater than 75,000.

In [145]:
print("Employees with salary > 75000:")
employees_df.filter((employees_df.salary) > 75000).show()


Employees with salary > 75000:
+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     7| Meera|   Finance| 82000|Bangalore|
|    10|Vikram|   Finance| 90000|    Delhi|
+------+------+----------+------+---------+



5. Calculate average salary.

In [146]:
from ast import alias
from pyspark.sql.functions import sum,avg,min,max
print("Average salary:")
employees_df.select(avg("salary").alias("average_salary")).show()

Average salary:
+--------------+
|average_salary|
+--------------+
|       69800.0|
+--------------+



6. Find highest paid employee.

In [147]:
print("Highest paid employee:")
employees_df.orderBy((employees_df.salary.desc())).show(1)

Highest paid employee:
+------+------+----------+------+-----+
|emp_id|  name|department|salary| city|
+------+------+----------+------+-----+
|    10|Vikram|   Finance| 90000|Delhi|
+------+------+----------+------+-----+
only showing top 1 row


7. Find lowest paid employee.

In [148]:
print("Lowest paid employee:")
employees_df.orderBy((employees_df.salary.asc())).show(1)

Lowest paid employee:
+------+-----+----------+------+------+
|emp_id| name|department|salary|  city|
+------+-----+----------+------+------+
|     5|Karan|        IT| 50000|Mumbai|
+------+-----+----------+------+------+
only showing top 1 row


8. Count employees per department.

In [149]:
print("Employee count per department:")
employees_df.groupBy("department").count().show()

Employee count per department:
+----------+-----+
|department|count|
+----------+-----+
|        HR|    3|
|   Finance|    3|
|        IT|    4|
+----------+-----+



9. Calculate average salary per department.

In [150]:
print("Average salary per department:")
employees_df.groupBy("department").agg(avg("salary").alias("avg_salary")).show()

Average salary per department:
+----------+------------------+
|department|        avg_salary|
+----------+------------------+
|        HR|59666.666666666664|
|   Finance|           84000.0|
|        IT|           66750.0|
+----------+------------------+



10. Find how many employees are in each city.

In [151]:
print("Employees in each city:")
employees_df.groupBy("city").count().show()

Employees in each city:
+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    2|
|   Mumbai|    1|
|    Delhi|    2|
|Hyderabad|    3|
+---------+-----+



11. Find the top 5 highest salaries.

In [152]:
print(" Top 5 highest salaries:")
employees_df.orderBy((employees_df.salary).desc()).show(5)

 Top 5 highest salaries:
+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|    10|Vikram|   Finance| 90000|    Delhi|
|     7| Meera|   Finance| 82000|Bangalore|
|     4| Priya|   Finance| 80000|Hyderabad|
|     3| Arjun|        IT| 75000|  Chennai|
|     8|  Ravi|        IT| 72000|Hyderabad|
+------+------+----------+------+---------+
only showing top 5 rows


12. Find employees working in Hyderabad with salary > 70k.

In [180]:
print("Hyderabad employees with salary > 70000:")
employees_df.filter((employees_df.city == "Hyderabad") & (employees_df.salary > 70000)).show()

Hyderabad employees with salary > 70000:
+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     4|Priya|   Finance| 80000|Hyderabad|
|     8| Ravi|        IT| 72000|Hyderabad|
+------+-----+----------+------+---------+



Dataset 3 — CSV (Sales Dataset)

Questions (Sales Analysis)

1. Calculate revenue per sale.

In [154]:
sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)

In [155]:
sales_df = sales_df.withColumn("revenue", sales_df.quantity * sales_df.price)
print("Revenue per sale:")
sales_df.select("sale_id", "emp_id", "product", "quantity", "price", "revenue").show()

Revenue per sale:
+-------+------+--------+--------+-----+-------+
|sale_id|emp_id| product|quantity|price|revenue|
+-------+------+--------+--------+-----+-------+
|      1|     1|  Laptop|       1|75000|  75000|
|      2|     2|   Mouse|       3|  500|   1500|
|      3|     3|Keyboard|       2| 1500|   3000|
|      4|     1| Monitor|       1|12000|  12000|
|      5|     4|  Laptop|       1|75000|  75000|
+-------+------+--------+--------+-----+-------+



2. Calculate total revenue.

In [156]:
print("Total revenue:")
sales_df.select(sum("revenue").alias("total_revenue")).show()

Total revenue:
+-------------+
|total_revenue|
+-------------+
|       166500|
+-------------+



3. Find revenue per product.

In [157]:
print("Revenue per product:")
sales_df.groupBy("product").agg(sum("revenue").alias("product_revenue")).show()


Revenue per product:
+--------+---------------+
| product|product_revenue|
+--------+---------------+
|  Laptop|         150000|
|   Mouse|           1500|
|Keyboard|           3000|
| Monitor|          12000|
+--------+---------------+



4. Find total quantity sold per product.

In [158]:
print("otal quantity sold per product:")
sales_df.groupBy("product").agg(sum("quantity").alias("total_quantity")).show()

otal quantity sold per product:
+--------+--------------+
| product|total_quantity|
+--------+--------------+
|  Laptop|             2|
|   Mouse|             3|
|Keyboard|             2|
| Monitor|             1|
+--------+--------------+



5. Find best selling product.

In [159]:
print("Best selling product:")
sales_df.groupBy("product") \
    .agg(sum("quantity").alias("total_quantity")) \
    .orderBy("total_quantity", ascending=False) \
    .show(1)

Best selling product:
+-------+--------------+
|product|total_quantity|
+-------+--------------+
|  Mouse|             3|
+-------+--------------+
only showing top 1 row


6. Find employee generating highest revenue.

In [160]:
print("Employee generating highest revenue:")
sales_df.groupBy("emp_id") \
    .agg(sum("revenue").alias("employee_revenue")) \
    .orderBy("employee_revenue",ascending=False) \
    .show(1)

Employee generating highest revenue:
+------+----------------+
|emp_id|employee_revenue|
+------+----------------+
|     1|           87000|
+------+----------------+
only showing top 1 row


7. Find average sale value.

In [161]:
print("Average sale value:")
sales_df.select(avg("revenue").alias("average_sale_value")).show()

Average sale value:
+------------------+
|average_sale_value|
+------------------+
|           33300.0|
+------------------+



8. Find products generating revenue above 100,000.

In [162]:
print("8. Products generating revenue above 100000:")
sales_df.groupBy("product") \
    .agg(sum("revenue").alias("product_revenue")) \
    .filter("product_revenue > 100000") \
    .show()

8. Products generating revenue above 100000:
+-------+---------------+
|product|product_revenue|
+-------+---------------+
| Laptop|         150000|
+-------+---------------+



Expected example output:

Laptop → 450000
Mouse → 7500
Keyboard → 15000
Monitor → 48000

Dataset 4 — JSON (Orders Dataset)

Questions (JSON Processing)

1. Load the JSON file.

In [163]:
orders_df = spark.read.option("multiline", "true").json("orders.json")
orders_df = orders_df.select(explode("orders").alias("order")).select("order.*")



2. Print all orders.

In [164]:
orders_df.show()

+------+---------+--------+--------+--------+
|amount|     city|customer|order_id| product|
+------+---------+--------+--------+--------+
| 75000|Hyderabad|   Rahul|       1|  Laptop|
|  1500|Bangalore|   Sneha|       2|   Mouse|
|  3000|  Chennai|   Arjun|       3|Keyboard|
| 75000|Hyderabad|   Priya|       4|  Laptop|
| 12000|   Mumbai|   Karan|       5| Monitor|
|  1000|Hyderabad|   Rahul|       6|   Mouse|
| 75000|Bangalore|   Sneha|       7|  Laptop|
|  3000|  Chennai|   Arjun|       8|Keyboard|
|  2000|Hyderabad|   Priya|       9|   Mouse|
| 12000|Hyderabad|   Rahul|      10| Monitor|
+------+---------+--------+--------+--------+



3. Count total orders.

In [165]:
print("Total orders:")
print(orders_df.count())

Total orders:
10


4. Calculate total sales amount.

In [166]:
print("Total sales:")
orders_df.select(sum("amount").alias("total_sales")).show()

Total sales:
+-----------+
|total_sales|
+-----------+
|     259500|
+-----------+



5. Find total spending per customer.

In [167]:
print("Spending per customer:")
orders_df.groupBy("customer") \
    .agg(sum("amount").alias("total_spending")) \
    .show()

Spending per customer:
+--------+--------------+
|customer|total_spending|
+--------+--------------+
|   Sneha|         76500|
|   Priya|         77000|
|   Rahul|         88000|
|   Arjun|          6000|
|   Karan|         12000|
+--------+--------------+



6. Find highest spending customer.

In [168]:
print("Top customer:")
orders_df.groupBy("customer") \
    .agg(sum("amount").alias("total_spending")) \
    .orderBy(desc("total_spending")) \
    .show(1)

Top customer:
+--------+--------------+
|customer|total_spending|
+--------+--------------+
|   Rahul|         88000|
+--------+--------------+
only showing top 1 row


7. Find total sales per product.

In [181]:
print("Total sales per product:")
orders_df.groupBy("product") \
    .agg(sum("amount").alias("total_sales")) \
    .show()

Total sales per product:
+--------+-----------+
| product|total_sales|
+--------+-----------+
|  Laptop|     225000|
|   Mouse|       4500|
|Keyboard|       6000|
| Monitor|      24000|
+--------+-----------+



8. Find customers from Hyderabad.

In [182]:
print("Customers from Hyderabad:")
orders_df.filter(col("city") == "Hyderabad") \
    .select("customer") \
    .distinct() \
    .show()

Customers from Hyderabad:
+--------+
|customer|
+--------+
|   Priya|
|   Rahul|
+--------+



9. Find orders with amount greater than 10,000.

In [171]:
print("High value orders:")
orders_df.filter(orders_df.amount > 10000).show()

High value orders:
+------+---------+--------+--------+-------+
|amount|     city|customer|order_id|product|
+------+---------+--------+--------+-------+
| 75000|Hyderabad|   Rahul|       1| Laptop|
| 75000|Hyderabad|   Priya|       4| Laptop|
| 12000|   Mumbai|   Karan|       5|Monitor|
| 75000|Bangalore|   Sneha|       7| Laptop|
| 12000|Hyderabad|   Rahul|      10|Monitor|
+------+---------+--------+--------+-------+



10. Count how many orders were placed in each city.

In [172]:
print("Orders per city:")
orders_df.groupBy("city").count().show()

Orders per city:
+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    2|
|   Mumbai|    1|
|Hyderabad|    5|
+---------+-----+



Final Combined Challenge

Using all datasets:

1. Join employees.csv and sales.csv using emp_id .

In [183]:
joined_df = employees_df.join(sales_df, on="emp_id", how="inner")

print("Joined Data:")
joined_df.show()


Joined Data:
+------+-----+----------+------+---------+-------+--------+--------+-----+-------+
|emp_id| name|department|salary|     city|sale_id| product|quantity|price|revenue|
+------+-----+----------+------+---------+-------+--------+--------+-----+-------+
|     1|Rahul|        IT| 70000|Hyderabad|      4| Monitor|       1|12000|  12000|
|     1|Rahul|        IT| 70000|Hyderabad|      1|  Laptop|       1|75000|  75000|
|     2|Sneha|        HR| 60000|Bangalore|      2|   Mouse|       3|  500|   1500|
|     3|Arjun|        IT| 75000|  Chennai|      3|Keyboard|       2| 1500|   3000|
|     4|Priya|   Finance| 80000|Hyderabad|      5|  Laptop|       1|75000|  75000|
+------+-----+----------+------+---------+-------+--------+--------+-----+-------+



2. Find total revenue generated by each employee.

In [174]:
employee_revenue = joined_df.groupBy("emp_id", "name") \
    .agg(sum("revenue").alias("total_revenue"))
employee_revenue.show()


+------+-----+-------------+
|emp_id| name|total_revenue|
+------+-----+-------------+
|     1|Rahul|        87000|
|     4|Priya|        75000|
|     2|Sneha|         1500|
|     3|Arjun|         3000|
+------+-----+-------------+



3. Find top 5 employees by sales.

In [175]:
employee_revenue.orderBy("total_revenue", ascending=False).show(5)

+------+-----+-------------+
|emp_id| name|total_revenue|
+------+-----+-------------+
|     1|Rahul|        87000|
|     4|Priya|        75000|
|     3|Arjun|         3000|
|     2|Sneha|         1500|
+------+-----+-------------+



4. Find department generating highest revenue.

In [176]:
department_revenue = joined_df.groupBy("department") \
    .agg(sum("revenue").alias("department_revenue"))
department_revenue.orderBy("department_revenue", ascending=False).show(1)

+----------+------------------+
|department|department_revenue|
+----------+------------------+
|        IT|             90000|
+----------+------------------+
only showing top 1 row


5. Save final result to: final_sales_report.csv

In [178]:
employee_revenue.orderBy("total_revenue", ascending=False) \
    .coalesce(1) \
    .write.mode("overwrite") \
    .option("header", True) \
    .csv("final_sales_report.csv")

Example output:

Employee Revenue
Rahul → 162000
Priya → 75000
Arjun → 4500